# Symbolic Eligibility Rule Encoding

This notebook converts selected ClinicalTrials.gov eligibility criteria into structured symbolic rules and evaluates synthetic osteosarcoma patients using three-valued logic: MET, NOT_MET, and UNKNOWN.

In [59]:
import json
from pathlib import Path
import pandas as pd

In [60]:
PROJECT_ROOT = Path("..")

TRIALS_DIR = PROJECT_ROOT / "data" / "trials"
PATIENTS_DIR = PROJECT_ROOT / "data" / "synthetic_patients"
RULES_DIR = PROJECT_ROOT / "data" / "rules"

RULES_DIR.mkdir(parents=True, exist_ok=True)

print("Trials directory:", TRIALS_DIR.resolve())
print("Patients directory:", PATIENTS_DIR.resolve())
print("Rules directory:", RULES_DIR.resolve())

Trials directory: D:\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant-Github\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant\data\trials
Patients directory: D:\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant-Github\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant\data\synthetic_patients
Rules directory: D:\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant-Github\Neuro-Symbolic-Pediatric-Oncology-Trial-Matching-Assistant\data\rules


In [61]:
five_trials = pd.read_csv(
    TRIALS_DIR / "five_proposed_trials.csv"
)

print("Selected trials:", len(five_trials))

five_trials[
    [
        "NCT Number",
        "Title",
        "Minimum Age",
        "Maximum Age"
    ]
]

Selected trials: 5


,NCT Number,Title,Minimum Age,Maximum Age
0,NCT07560826,The CATSINDO Trial - Clinical and Translationa...,7 Years,22 Years
1,NCT07144254,Tegavivint With Gemcitabine in Patients With R...,1 Year,30 Years
2,NCT05590572,A Study of Sulfatinib on Relapsed or Refractor...,2 Years,25 Years
3,NCT06235125,Safety and Feasibility of Intraoperative Visua...,6 Years,17 Years
4,NCT02013336,Phase 1 Dose-escalating Study of MM-398 (Irino...,12 Months,20 Years


In [62]:
with open(
    PATIENTS_DIR / "synthetic_patients.json",
    "r",
    encoding="utf-8"
) as f:
    synthetic_patients = json.load(f)

print("Synthetic patients loaded:", len(synthetic_patients))

for patient in synthetic_patients:
    print(
        patient["patient_id"],
        "-",
        patient["age_years"],
        "-",
        patient["diagnosis"]["disease_status"]
    )

Synthetic patients loaded: 8
P001 - 15 - newly_diagnosed
P002 - 16 - newly_diagnosed
P003 - 17 - relapsed
P004 - 14 - refractory
P005 - 12 - refractory
P006 - 13 - metastatic
P007 - 10 - recurrent
P008 - 19 - relapsed


In [63]:
trial_07560826 = five_trials[
    five_trials["NCT Number"] == "NCT07560826"
].iloc[0]

print("Trial:")
print(trial_07560826["Title"])

print("\nEligibility Criteria:")
print(trial_07560826["Eligibility Text"])

Trial:
The CATSINDO Trial - Clinical and Translational Study in Newly Diagnosed Osteosarcoma

Eligibility Criteria:
Pre-Enrollment Criteria:

1. Suspected diagnosis of high-grade osteosarcoma based on clinical and radiographic findings.
2. Informed consent (and assent, if applicable) obtained, per institutional guidelines.
3. Participants must be ≤ 22 years of age at the time of consent.

Inclusion Criteria:

1. Participants with localized or metastatic high-grade osteosarcoma.
2. Participants must be ≤ 22 years of age at the time of consent.
3. Participants must have a body surface area of greater than or equal to 0.8 m2.
4. Participants receiving or planning to receive induction/neoadjuvant MAP chemotherapy with HD-MTX given at the standard dose of 12 gm/m2 (maximum 20 gm).

   NOTE: Participants may be participating on other clinical studies such as the COG trial AOST2032 or any other clinical trial as long as their treatment includes MAP chemotherapy with the standard HD-MTX dose o

## Symbolic Rule Representation

Each eligibility criterion is represented as a structured rule containing:
- a unique rule ID
- the patient field to inspect
- an operator
- the expected value or threshold
- the original criterion text
- whether the rule is an inclusion or exclusion criterion

The rule engine will later evaluate each rule as MET, NOT_MET, or UNKNOWN.

In [64]:
def make_rule(
    rule_id,
    field,
    operator,
    value,
    criterion_type,
    description
):
    return {
        "rule_id": rule_id,
        "field": field,
        "operator": operator,
        "value": value,
        "criterion_type": criterion_type,
        "description": description
    }

## NCT07560826 — CATSINDO Symbolic Rules

These rules encode the machine-checkable eligibility requirements from the ClinicalTrials.gov eligibility text.

In [65]:
rules_NCT07560826 = [

    make_rule(
        "NCT07560826_R01",
        "age_years",
        "<=",
        22,
        "inclusion",
        "Participant must be 22 years of age or younger."
    ),

    make_rule(
        "NCT07560826_R02",
        "diagnosis.cancer_type",
        "==",
        "osteosarcoma",
        "inclusion",
        "Participant must have osteosarcoma."
    ),

    make_rule(
        "NCT07560826_R03",
        "diagnosis.grade",
        "==",
        "high-grade",
        "inclusion",
        "Participant must have high-grade osteosarcoma."
    ),

    make_rule(
        "NCT07560826_R04",
        "diagnosis.disease_extent",
        "in",
        ["localized", "metastatic"],
        "inclusion",
        "Localized or metastatic high-grade osteosarcoma is permitted."
    ),

    make_rule(
        "NCT07560826_R05",
        "anthropometrics.body_surface_area_m2",
        ">=",
        0.8,
        "inclusion",
        "Body surface area must be at least 0.8 m2."
    ),

    make_rule(
        "NCT07560826_R06",
        "current_treatment.map_chemotherapy",
        "==",
        True,
        "inclusion",
        "Participant must be receiving or planning induction/neoadjuvant MAP chemotherapy."
    ),

    make_rule(
        "NCT07560826_R07",
        "current_treatment.methotrexate",
        "==",
        True,
        "inclusion",
        "MAP treatment must include high-dose methotrexate."
    ),

    make_rule(
        "NCT07560826_R08",
        "current_treatment.methotrexate_dose_g_m2",
        "==",
        12,
        "inclusion",
        "High-dose methotrexate is given at the standard dose of 12 g/m2."
    ),

    make_rule(
        "NCT07560826_R09",
        "laboratory_values.bilirubin_uln_ratio",
        "<=",
        1.5,
        "inclusion",
        "Total bilirubin must be no more than 1.5 times the upper limit of normal."
    ),

    make_rule(
        "NCT07560826_R10",
        "laboratory_values.alt_u_l",
        "<=",
        135,
        "inclusion",
        "ALT must be no more than 135 U/L."
    ),

    make_rule(
        "NCT07560826_R11",
        "laboratory_values.gfr_ml_min_1_73m2",
        ">=",
        70,
        "inclusion",
        "GFR or creatinine clearance must be at least 70 mL/min/1.73m2."
    ),

    make_rule(
        "NCT07560826_R12",
        "laboratory_values.anc_per_ul",
        ">=",
        1000,
        "inclusion",
        "Absolute neutrophil count must be at least 1000/uL."
    ),

    make_rule(
        "NCT07560826_R13",
        "laboratory_values.platelets_per_ul",
        ">=",
        100000,
        "inclusion",
        "Platelet count must be at least 100000/uL."
    ),

    make_rule(
        "NCT07560826_R14",
        "laboratory_values.hemoglobin_g_dl",
        ">=",
        8.0,
        "inclusion",
        "Hemoglobin must be at least 8.0 g/dL."
    ),

    make_rule(
        "NCT07560826_R15",
        "cardiac.lvef_percent",
        ">=",
        50,
        "inclusion",
        "Left ventricular ejection fraction must be at least 50 percent."
    ),

    make_rule(
        "NCT07560826_R16",
        "current_treatment.second_cycle_started",
        "==",
        False,
        "inclusion",
        "Participant must enroll before the second cycle of chemotherapy."
    ),

    make_rule(
        "NCT07560826_R17",
        "other_clinical_facts.pregnant",
        "==",
        False,
        "exclusion",
        "Pregnant participants are excluded."
    ),

    make_rule(
        "NCT07560826_R18",
        "other_clinical_facts.breastfeeding",
        "==",
        False,
        "exclusion",
        "Breastfeeding participants are excluded."
    )
]

In [66]:
print("Number of encoded rules:", len(rules_NCT07560826))

pd.DataFrame(rules_NCT07560826)

Number of encoded rules: 18


,rule_id,field,operator,value,criterion_type,description
0,NCT07560826_R01,age_years,<=,22,inclusion,Participant must be 22 years of age or younger.
1,NCT07560826_R02,diagnosis.cancer_type,==,osteosarcoma,inclusion,Participant must have osteosarcoma.
2,NCT07560826_R03,diagnosis.grade,==,high-grade,inclusion,Participant must have high-grade osteosarcoma.
3,NCT07560826_R04,diagnosis.disease_extent,in,"[localized, metastatic]",inclusion,Localized or metastatic high-grade osteosarcom...
4,NCT07560826_R05,anthropometrics.body_surface_area_m2,>=,0.8,inclusion,Body surface area must be at least 0.8 m2.
5,NCT07560826_R06,current_treatment.map_chemotherapy,==,True,inclusion,Participant must be receiving or planning indu...
6,NCT07560826_R07,current_treatment.methotrexate,==,True,inclusion,MAP treatment must include high-dose methotrex...
7,NCT07560826_R08,current_treatment.methotrexate_dose_g_m2,==,12,inclusion,High-dose methotrexate is given at the standar...
8,NCT07560826_R09,laboratory_values.bilirubin_uln_ratio,<=,1.5,inclusion,Total bilirubin must be no more than 1.5 times...
9,NCT07560826_R10,laboratory_values.alt_u_l,<=,135,inclusion,ALT must be no more than 135 U/L.


In [67]:
def get_nested_value(data, field_path):
    keys = field_path.split(".")
    value = data

    for key in keys:
        if not isinstance(value, dict) or key not in value:
            return None

        value = value[key]

    return value

In [68]:
print(
    get_nested_value(
        synthetic_patients[0],
        "age_years"
    )
)

print(
    get_nested_value(
        synthetic_patients[0],
        "laboratory_values.anc_per_ul"
    )
)

print(
    get_nested_value(
        synthetic_patients[0],
        "cardiac.lvef_percent"
    )
)

15
1850
61


## Three-Valued Symbolic Rule Evaluation

Each rule is evaluated using three-valued logic:

- MET: the patient fact satisfies the eligibility rule
- NOT_MET: the patient fact contradicts the eligibility rule
- UNKNOWN: the required patient fact is missing

In [69]:
def evaluate_rule(patient, rule):
    operator = rule["operator"]
    expected_value = rule["value"]

    if operator == "count_true_gte":
        fields = rule["field"]

        values = [
            get_nested_value(patient, field)
            for field in fields
        ]

        true_count = sum(
            value is True
            for value in values
        )

        unknown_count = sum(
            value is None
            for value in values
        )

        if true_count >= expected_value:
            return "MET"

        if true_count + unknown_count >= expected_value:
            return "UNKNOWN"

        return "NOT_MET"

    patient_value = get_nested_value(
        patient,
        rule["field"]
    )

    if patient_value is None:
        return "UNKNOWN"

    try:
        if operator == "==":
            result = patient_value == expected_value

        elif operator == "!=":
            result = patient_value != expected_value

        elif operator == ">=":
            result = patient_value >= expected_value

        elif operator == "<=":
            result = patient_value <= expected_value

        elif operator == ">":
            result = patient_value > expected_value

        elif operator == "<":
            result = patient_value < expected_value

        elif operator == "in":
            result = patient_value in expected_value

        else:
            raise ValueError(
                f"Unsupported operator: {operator}"
            )

    except (TypeError, ValueError):
        return "UNKNOWN"

    return "MET" if result else "NOT_MET"

In [70]:
patients_by_id = {
    patient["patient_id"]: patient
    for patient in synthetic_patients
}

P001 = patients_by_id["P001"]
P002 = patients_by_id["P002"]
P003 = patients_by_id["P003"]
P004 = patients_by_id["P004"]
P005 = patients_by_id["P005"]
P006 = patients_by_id["P006"]
P007 = patients_by_id["P007"]
P008 = patients_by_id["P008"]

print("Patient references created successfully.")

Patient references created successfully.


In [71]:
anc_rule = next(
    rule
    for rule in rules_NCT07560826
    if rule["rule_id"] == "NCT07560826_R12"
)

print(
    "P001 ANC:",
    get_nested_value(P001, anc_rule["field"]),
    "→",
    evaluate_rule(P001, anc_rule)
)

print(
    "P002 ANC:",
    get_nested_value(P002, anc_rule["field"]),
    "→",
    evaluate_rule(P002, anc_rule)
)

P001 ANC: 1850 → MET
P002 ANC: 620 → NOT_MET


In [72]:
def evaluate_trial(patient, rules):
    results = []

    for rule in rules:
        if isinstance(rule["field"], list):
            patient_value = {
                field: get_nested_value(patient, field)
                for field in rule["field"]
            }
        else:
            patient_value = get_nested_value(
                patient,
                rule["field"]
            )

        status = evaluate_rule(
            patient,
            rule
        )

        results.append({
            "patient_id": patient["patient_id"],
            "rule_id": rule["rule_id"],
            "criterion_type": rule["criterion_type"],
            "field": rule["field"],
            "patient_value": patient_value,
            "operator": rule["operator"],
            "expected_value": rule["value"],
            "status": status,
            "description": rule["description"]
        })

    return results

In [73]:
def determine_overall_status(results):
    statuses = [
        result["status"]
        for result in results
    ]

    if "NOT_MET" in statuses:
        return "NOT_MET"

    if "UNKNOWN" in statuses:
        return "UNKNOWN"

    return "MET"

## Validate NCT07560826 With Synthetic Patients

P001 represents a patient expected to meet all encoded eligibility requirements.

P002 represents a patient expected not to meet the trial because of inadequate bone marrow and cardiac function.

In [74]:
P001_results = evaluate_trial(
    P001,
    rules_NCT07560826
)

P001_results_df = pd.DataFrame(P001_results)

P001_results_df[
    [
        "rule_id",
        "field",
        "patient_value",
        "operator",
        "expected_value",
        "status"
    ]
]

,rule_id,field,patient_value,operator,expected_value,status
0,NCT07560826_R01,age_years,15,<=,22,MET
1,NCT07560826_R02,diagnosis.cancer_type,osteosarcoma,==,osteosarcoma,MET
2,NCT07560826_R03,diagnosis.grade,high-grade,==,high-grade,MET
3,NCT07560826_R04,diagnosis.disease_extent,localized,in,"[localized, metastatic]",MET
4,NCT07560826_R05,anthropometrics.body_surface_area_m2,1.55,>=,0.8,MET
5,NCT07560826_R06,current_treatment.map_chemotherapy,True,==,True,MET
6,NCT07560826_R07,current_treatment.methotrexate,True,==,True,MET
7,NCT07560826_R08,current_treatment.methotrexate_dose_g_m2,12,==,12,MET
8,NCT07560826_R09,laboratory_values.bilirubin_uln_ratio,0.8,<=,1.5,MET
9,NCT07560826_R10,laboratory_values.alt_u_l,52,<=,135,MET


In [75]:
P002_results = evaluate_trial(
    P002,
    rules_NCT07560826
)

P002_results_df = pd.DataFrame(P002_results)

P002_results_df[
    [
        "rule_id",
        "field",
        "patient_value",
        "operator",
        "expected_value",
        "status"
    ]
]

,rule_id,field,patient_value,operator,expected_value,status
0,NCT07560826_R01,age_years,16,<=,22,MET
1,NCT07560826_R02,diagnosis.cancer_type,osteosarcoma,==,osteosarcoma,MET
2,NCT07560826_R03,diagnosis.grade,high-grade,==,high-grade,MET
3,NCT07560826_R04,diagnosis.disease_extent,metastatic,in,"[localized, metastatic]",MET
4,NCT07560826_R05,anthropometrics.body_surface_area_m2,1.48,>=,0.8,MET
5,NCT07560826_R06,current_treatment.map_chemotherapy,True,==,True,MET
6,NCT07560826_R07,current_treatment.methotrexate,True,==,True,MET
7,NCT07560826_R08,current_treatment.methotrexate_dose_g_m2,12,==,12,MET
8,NCT07560826_R09,laboratory_values.bilirubin_uln_ratio,1.1,<=,1.5,MET
9,NCT07560826_R10,laboratory_values.alt_u_l,68,<=,135,MET


In [76]:
print(
    "P001 overall:",
    determine_overall_status(P001_results)
)

print(
    "P002 overall:",
    determine_overall_status(P002_results)
)

P001 overall: MET
P002 overall: NOT_MET


In [77]:
P002_results_df[
    [
        "rule_id",
        "field",
        "patient_value",
        "expected_value",
        "status"
    ]
]

,rule_id,field,patient_value,expected_value,status
0,NCT07560826_R01,age_years,16,22,MET
1,NCT07560826_R02,diagnosis.cancer_type,osteosarcoma,osteosarcoma,MET
2,NCT07560826_R03,diagnosis.grade,high-grade,high-grade,MET
3,NCT07560826_R04,diagnosis.disease_extent,metastatic,"[localized, metastatic]",MET
4,NCT07560826_R05,anthropometrics.body_surface_area_m2,1.48,0.8,MET
5,NCT07560826_R06,current_treatment.map_chemotherapy,True,True,MET
6,NCT07560826_R07,current_treatment.methotrexate,True,True,MET
7,NCT07560826_R08,current_treatment.methotrexate_dose_g_m2,12,12,MET
8,NCT07560826_R09,laboratory_values.bilirubin_uln_ratio,1.1,1.5,MET
9,NCT07560826_R10,laboratory_values.alt_u_l,68,135,MET


In [78]:
print("P001 status counts:")
print(P001_results_df["status"].value_counts())

print("\nP002 status counts:")
print(P002_results_df["status"].value_counts())

P001 status counts:
status
MET    18
Name: count, dtype: int64

P002 status counts:
status
MET        14
NOT_MET     4
Name: count, dtype: int64


## NCT07144254 — Tegavivint With Gemcitabine

The next trial evaluates patients with relapsed or refractory osteosarcoma. Its encoded rules include prior frontline therapy, performance status, treatment washout periods, organ function requirements, and relevant exclusions.

In [79]:
trial_07144254 = five_trials[
    five_trials["NCT Number"] == "NCT07144254"
].iloc[0]

print("Trial:")
print(trial_07144254["Title"])

print("\nEligibility Criteria:")
print(trial_07144254["Eligibility Text"])

Trial:
Tegavivint With Gemcitabine in Patients With Relapsed or Refractory Osteosarcoma

Eligibility Criteria:
Inclusion Criteria:

Diagnosis:

Participants must have had histologic verification of osteosarcoma at original diagnosis or relapse.

* All participants with relapsed or refractory osteosarcoma are eligible, provided they received front-line treatment with a regimen that contained at least 3 of the following agents: methotrexate, doxorubicin, cisplatin, and ifosfamide

  -Disease Status:
* Dose Escalation: Participants must have either measurable or evaluable disease per RECIST.Note: Participants with no evidence of disease on imaging (e.g., following pulmonary metastasectomy) are not eligible during the dose escalation phase.
* Dose Expansion: Participants with measurable or evaluable disease per RECIST and those with no evidence of disease on imaging following pulmonary metastasectomy are eligible during the dose expansion phase.

  -Performance Level: Participants must hav

## NCT07144254 — Tegavivint + Gemcitabine Symbolic Rules

These rules represent machine-checkable eligibility criteria for patients with relapsed or refractory osteosarcoma.

In [80]:
rules_NCT07144254 = [

    make_rule(
        "NCT07144254_R01",
        "diagnosis.histologic_confirmation",
        "==",
        True,
        "inclusion",
        "Osteosarcoma must have histologic verification."
    ),

    make_rule(
        "NCT07144254_R02",
        "diagnosis.disease_status",
        "in",
        ["relapsed", "refractory"],
        "inclusion",
        "Participant must have relapsed or refractory osteosarcoma."
    ),

    make_rule(
        "NCT07144254_R03",
        [
            "prior_treatment.methotrexate",
            "prior_treatment.doxorubicin",
            "prior_treatment.cisplatin",
            "prior_treatment.ifosfamide"
        ],
        "count_true_gte",
        3,
        "inclusion",
        "Front-line therapy must have included at least 3 of methotrexate, doxorubicin, cisplatin, and ifosfamide."
    ),

    make_rule(
        "NCT07144254_R04",
        "diagnosis.measurable_disease",
        "==",
        True,
        "inclusion",
        "Participant must have measurable or evaluable disease for the modeled cohort."
    ),

    make_rule(
        "NCT07144254_R05",
        "performance_status.score",
        ">=",
        60,
        "inclusion",
        "Lansky or Karnofsky performance score must be at least 60."
    ),

    make_rule(
        "NCT07144254_R06",
        "treatment_timing_days.last_chemotherapy",
        ">=",
        14,
        "inclusion",
        "At least 14 days must have passed since myelosuppressive chemotherapy."
    ),

    make_rule(
        "NCT07144254_R07",
        "treatment_timing_days.last_major_surgery",
        ">=",
        14,
        "inclusion",
        "At least 2 weeks must have passed since major surgery."
    ),

    make_rule(
        "NCT07144254_R08",
        "laboratory_values.anc_per_ul",
        ">=",
        750,
        "inclusion",
        "ANC must be at least 750/mm3."
    ),

    make_rule(
        "NCT07144254_R09",
        "laboratory_values.platelets_per_ul",
        ">=",
        75000,
        "inclusion",
        "Platelet count must be at least 75000/mm3."
    ),

    make_rule(
        "NCT07144254_R10",
        "laboratory_values.gfr_ml_min_1_73m2",
        ">=",
        70,
        "inclusion",
        "Creatinine clearance or GFR must be at least 70 mL/min/1.73m2."
    ),

    make_rule(
        "NCT07144254_R11",
        "laboratory_values.bilirubin_uln_ratio",
        "<=",
        1.5,
        "inclusion",
        "Bilirubin must be no more than 1.5 times ULN."
    ),

    make_rule(
        "NCT07144254_R12",
        "laboratory_values.alt_uln_ratio",
        "<=",
        5,
        "inclusion",
        "ALT must be no more than 5 times ULN."
    ),

    make_rule(
        "NCT07144254_R13",
        "pulmonary.dyspnea_at_rest",
        "==",
        False,
        "inclusion",
        "Participant must not have dyspnea at rest."
    ),

    make_rule(
        "NCT07144254_R14",
        "pulmonary.exercise_intolerance",
        "==",
        False,
        "inclusion",
        "Participant must not have exercise intolerance."
    ),

    make_rule(
        "NCT07144254_R15",
        "pulmonary.oxygen_requirement",
        "==",
        False,
        "inclusion",
        "Participant must not require supplemental oxygen."
    ),

    make_rule(
        "NCT07144254_R16",
        "pulmonary.oxygen_saturation_percent",
        ">",
        93,
        "inclusion",
        "Room-air oxygen saturation must be greater than 93 percent."
    ),

    make_rule(
        "NCT07144254_R17",
        "cardiac.qtc_ms",
        "<=",
        470,
        "inclusion",
        "QTc must be 470 ms or less."
    ),

    make_rule(
        "NCT07144254_R18",
        "other_clinical_facts.active_infection",
        "==",
        False,
        "exclusion",
        "Active uncontrolled infection is exclusionary."
    ),

    make_rule(
        "NCT07144254_R19",
        "other_clinical_facts.prior_tegavivint",
        "==",
        False,
        "exclusion",
        "Previous tegavivint treatment is not allowed."
    )
]

In [81]:
print(
    "Number of encoded rules:",
    len(rules_NCT07144254)
)

pd.DataFrame(rules_NCT07144254)

Number of encoded rules: 19


,rule_id,field,operator,value,criterion_type,description
0,NCT07144254_R01,diagnosis.histologic_confirmation,==,True,inclusion,Osteosarcoma must have histologic verification.
1,NCT07144254_R02,diagnosis.disease_status,in,"[relapsed, refractory]",inclusion,Participant must have relapsed or refractory o...
2,NCT07144254_R03,"[prior_treatment.methotrexate, prior_treatment...",count_true_gte,3,inclusion,Front-line therapy must have included at least...
3,NCT07144254_R04,diagnosis.measurable_disease,==,True,inclusion,Participant must have measurable or evaluable ...
4,NCT07144254_R05,performance_status.score,>=,60,inclusion,Lansky or Karnofsky performance score must be ...
5,NCT07144254_R06,treatment_timing_days.last_chemotherapy,>=,14,inclusion,At least 14 days must have passed since myelos...
6,NCT07144254_R07,treatment_timing_days.last_major_surgery,>=,14,inclusion,At least 2 weeks must have passed since major ...
7,NCT07144254_R08,laboratory_values.anc_per_ul,>=,750,inclusion,ANC must be at least 750/mm3.
8,NCT07144254_R09,laboratory_values.platelets_per_ul,>=,75000,inclusion,Platelet count must be at least 75000/mm3.
9,NCT07144254_R10,laboratory_values.gfr_ml_min_1_73m2,>=,70,inclusion,Creatinine clearance or GFR must be at least 7...


## Validate NCT07144254 With Synthetic Patients

P003 represents a strong match expected to meet every encoded requirement.

P004 represents a patient with several missing clinical values. Those missing facts should produce UNKNOWN results rather than incorrectly producing MET or NOT_MET.

In [82]:
P003_results = evaluate_trial(
    P003,
    rules_NCT07144254
)

P003_results_df = pd.DataFrame(P003_results)

P003_results_df[
    [
        "rule_id",
        "field",
        "patient_value",
        "expected_value",
        "status"
    ]
]

,rule_id,field,patient_value,expected_value,status
0,NCT07144254_R01,diagnosis.histologic_confirmation,True,True,MET
1,NCT07144254_R02,diagnosis.disease_status,relapsed,"[relapsed, refractory]",MET
2,NCT07144254_R03,"[prior_treatment.methotrexate, prior_treatment...","{'prior_treatment.methotrexate': True, 'prior_...",3,MET
3,NCT07144254_R04,diagnosis.measurable_disease,True,True,MET
4,NCT07144254_R05,performance_status.score,80,60,MET
5,NCT07144254_R06,treatment_timing_days.last_chemotherapy,28,14,MET
6,NCT07144254_R07,treatment_timing_days.last_major_surgery,35,14,MET
7,NCT07144254_R08,laboratory_values.anc_per_ul,1420,750,MET
8,NCT07144254_R09,laboratory_values.platelets_per_ul,118000,75000,MET
9,NCT07144254_R10,laboratory_values.gfr_ml_min_1_73m2,88,70,MET


In [83]:
print(
    "P003 overall:",
    determine_overall_status(P003_results)
)

print(
    P003_results_df["status"].value_counts()
)

P003 overall: MET
status
MET    19
Name: count, dtype: int64


In [84]:
P004_results = evaluate_trial(
    P004,
    rules_NCT07144254
)

P004_results_df = pd.DataFrame(P004_results)

P004_results_df[
    [
        "rule_id",
        "field",
        "patient_value",
        "expected_value",
        "status"
    ]
]

,rule_id,field,patient_value,expected_value,status
0,NCT07144254_R01,diagnosis.histologic_confirmation,True,True,MET
1,NCT07144254_R02,diagnosis.disease_status,refractory,"[relapsed, refractory]",MET
2,NCT07144254_R03,"[prior_treatment.methotrexate, prior_treatment...","{'prior_treatment.methotrexate': True, 'prior_...",3,MET
3,NCT07144254_R04,diagnosis.measurable_disease,True,True,MET
4,NCT07144254_R05,performance_status.score,70,60,MET
5,NCT07144254_R06,treatment_timing_days.last_chemotherapy,None,14,UNKNOWN
6,NCT07144254_R07,treatment_timing_days.last_major_surgery,28,14,MET
7,NCT07144254_R08,laboratory_values.anc_per_ul,1100,750,MET
8,NCT07144254_R09,laboratory_values.platelets_per_ul,92000,75000,MET
9,NCT07144254_R10,laboratory_values.gfr_ml_min_1_73m2,None,70,UNKNOWN


In [85]:
print(
    "P004 overall:",
    determine_overall_status(P004_results)
)

print(
    P004_results_df["status"].value_counts()
)

P004 overall: UNKNOWN
status
MET        15
UNKNOWN     4
Name: count, dtype: int64
